<a href="https://colab.research.google.com/github/SwRI-IDEA-Lab/butterflai/blob/development%2Fjhamilton/weeks/week_09/09d_conditioned_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 09 (evaluation): conditional sampling and distributional verification

This notebook is the evaluation half of Week 09. It assumes
`09c_conditioned_train.ipynb` has been run end-to-end and has produced
`ckpt_conditional.ckpt` in this directory. It *also* assumes Week 08 has
produced `ckpt_full.ckpt` (the unconditional t-aware model) — the
unconditional model is the natural baseline against which we measure
whether the conditioning machinery actually does anything useful at the
distributional level.

The headline value-add metric — `compute_global_nll` evaluated on the
held-out validation split, classical alone vs. classical + conditional
residuals — lives in its own notebook. *This* notebook answers the prior
question: does the conditional model produce samples that are sensitive
to the conditioning in the way we expect, and do those samples look
like training residuals at the distributional level?

The Week 09 evaluation tasks pick up where Week 08's Task 44 left off:

- **Load** both checkpoints and re-verify the t-sensitivity *and*
  cond-sensitivity of the loaded conditional model (the analogue of Week
  08's "redo Task 39 on the loaded model" defence).
- **Task 53**: sample residuals from the conditional model with a small
  set of distinct conditioning vectors, visualize how the samples
  qualitatively shift as `cond` varies.
- **Task 54**: distributional verification — for each validation window,
  sample N residuals at that window's conditioning, then compare the
  aggregate sampled distribution against the held-out validation
  residuals. The unconditional model serves as the reference baseline:
  does the conditional model do *better* at matching the validation
  distribution, or has the conditioning machinery learned nothing useful
  at this scale?


In [1]:
# To Keep - necessary for all notebooks
import os, subprocess, sys

# Keep this path if working in Colab
repo_path = "/content/butterflai"

# Use this path if working locally
# repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


  Installing from /content/butterflai/requirements.txt...

🦋 ButterflAI environment ready
   Runtime  : Google Colab
   Device   : cpu
   Seed     : 42


{'in_colab': True,
 'device': device(type='cpu'),
 'seed': 42,
 'drive_mounted': False,
 'data_path': None}

In [2]:
# Keep - to port in Google Drive for the checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Define the paths where your checkpoints are located within Google Drive.
# These paths are based on your clarifications.
UNCONDITIONAL_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab_Checkpoints/week_08"
CONDITIONAL_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab_Checkpoints/week_09"

# Ensure the checkpoint paths exist, if not, create them. (Optional, if you know they exist)
os.makedirs(UNCONDITIONAL_CHECKPOINT_PATH, exist_ok=True)
os.makedirs(CONDITIONAL_CHECKPOINT_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After executing the above cell and mounting your drive, please ensure that `CHECKPOINT_PATH` points to the directory containing your `ckpt_full.ckpt` and `ckpt_conditional.ckpt` files. You might need to adjust the path `"/content/drive/MyDrive/checkpoints"` if your files are in a different location.

In [3]:
#Keep - to import packages and pytorch-lightning

#%load_ext autoreload
#%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat

!pip install pytorch-lightning

In [11]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
import math # Added math import for timestep_embedding

from einops import repeat

# Get the full path to conditioned_infrastructure.py
conditioned_infrastructure_path = os.path.join(repo_path, 'weeks', 'week_09', 'conditioned_infrastructure.py')

# Construct the full, corrected `conditioned_infrastructure.py` content as a string.
# Explicitly ensure no leading whitespace within the triple-quoted string that would cause IndentationError.
conditioned_file_content = """import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
import math # Added math import for timestep_embedding

from einops import repeat

# Assuming these are available from unconditioned_infrastructure
from unconditioned_infrastructure import make_cosine_schedule # This will be imported in the main notebook so it's fine

# Added sinusoidal timestep embedding function to match checkpoint's expected input dimension for time_emb.mlp.0.weight
def _timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class ConditionalResidualDataset(torch.utils.data.Dataset):
    def __init__(self, r_norm: np.ndarray, cond_norm: np.ndarray):
        super().__init__()
        assert len(r_norm) == len(cond_norm)
        self.r_norm  = torch.from_numpy(r_norm).float()
        self.cond_norm = torch.from_numpy(cond_norm).float()

    def __len__(self):
        return len(self.r_norm)

    def __getitem__(self, idx):
        return self.r_norm[idx], self.cond_norm[idx]


class ConditionalDiffusionMLP(
    nn.Module
):
    def __init__(self, data_dim=15, cond_dim=4, hidden_dim=64, t_embed_dim=16, t_hidden_dim=32, n_layers=3):
        super().__init__()
        self.data_dim = data_dim
        self.cond_dim = cond_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        # Based on the size mismatch error, the input to the time embedding MLP should be 64-dimensional
        self.t_input_dim = 64

        # Timestep embedding (matches model.time_emb.mlp in checkpoint - ModuleDict wrapper)
        self.time_emb = nn.ModuleDict({
            'mlp': nn.Sequential(
                nn.Linear(self.t_input_dim, t_hidden_dim),
                nn.ReLU(),
                nn.Linear(t_hidden_dim, t_embed_dim)
            )
        })

        # Main MLP (matches model.mlp in checkpoint)
        layers = [
            nn.Linear(data_dim + cond_dim + t_embed_dim, hidden_dim),
            nn.ReLU()
        ]
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim, data_dim)) # Output layer
        self.mlp = nn.Sequential(*layers)

    def forward(self, r_t: torch.Tensor, t: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        # Generate timestep embedding using the helper function
        t_embed_input = _timestep_embedding(t, self.t_input_dim)

        # Timestep embedding through the MLP
        t_emb = self.time_emb['mlp'](t_embed_input)

        # Concatenate r_t, cond, and t_emb
        x = torch.cat([r_t, cond, t_emb], dim=-1)

        # Pass through the main MLP
        return self.mlp(x)


class ConditionalDiffusionLightning(pl.LightningModule):
    def __init__(self, model: ConditionalDiffusionMLP, alpha: np.ndarray, sigma: np.ndarray,
                 bin_means: np.ndarray, bin_stds: np.ndarray,
                 cond_means: np.ndarray, cond_stds: np.ndarray):
        super().__init__()
        self.model = model

        self.register_buffer("alpha", torch.from_numpy(alpha).float())
        self.register_buffer("sigma", torch.from_numpy(sigma).float())
        self.register_buffer("_alpha_bar", torch.from_numpy(np.sqrt(alpha)).float())

        self.register_buffer("bin_means", torch.from_numpy(bin_means).float())
        self.register_buffer("bin_stds", torch.from_numpy(bin_stds).float())

        self.register_buffer("cond_means", torch.from_numpy(cond_means).float())
        self.register_buffer("cond_stds", torch.from_numpy(cond_stds).float())

    def training_step(self, batch, batch_idx):
        r0, cond = batch
        N = len(r0)

        # ── noise process ───────────────────────────────────────────────────
        t = torch.randint(0, len(self.alpha), (N,), device=r0.device)

        epsilon = torch.randn_like(r0)
        r_t = r0 * self.alpha[t].unsqueeze(-1) + epsilon * self.sigma[t].unsqueeze(-1)

        # ── predict noise from noisy residual and conditioning ──────────────
        epsilon_pred = self.model(r_t, t, cond)
        loss = F.mse_loss(epsilon_pred, epsilon)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

    # For validation/testing, you might want a different step function
    def validation_step(self, batch, batch_idx):
        r0, cond = batch
        N = len(r0)

        t = torch.randint(0, len(self.alpha), (N,), device=r0.device)

        epsilon = torch.randn_like(r0)
        r_t = r0 * self.alpha[t].unsqueeze(-1) + epsilon * self.sigma[t].unsqueeze(-1)

        epsilon_pred = self.model(r_t, t, cond)
        loss = F.mse_loss(epsilon_pred, epsilon)
        self.log("val_loss", loss)
        return loss

def sample_conditional(model: ConditionalDiffusionLightning, cond: torch.Tensor, device: torch.device) -> torch.Tensor:
    N = len(cond)
    data_dim = model.model.data_dim # Assuming data_dim is accessible from the inner model

    # Start with pure noise
    r_t = torch.randn(N, data_dim, device=device)

    # Reverse process
    for t_idx in reversed(range(len(model.alpha))):
        t = torch.full((N,), t_idx, dtype=torch.long, device=device)

        # Predict noise
        epsilon_pred = model.model(r_t, t, cond)

        # Denoise step based on DDPM formula (using the _alpha_bar from the Lightning module)
        alpha_t = model.alpha[t].unsqueeze(-1)
        alpha_t_prev = model.alpha[t-1].unsqueeze(-1) if t_idx > 0 else torch.tensor(1.0, device=device).unsqueeze(-1)
        sigma_t = model.sigma[t].unsqueeze(-1)
        sigma_t_prev = model.sigma[t-1].unsqueeze(-1) if t_idx > 0 else torch.tensor(0.0, device=device).unsqueeze(-1)

        # Simplified reverse step (common in many implementations, directly predicting r0 and then re-noising)
        pred_r0 = (r_t - sigma_t * epsilon_pred) / alpha_t
        # Clamp pred_r0 to prevent numerical instability leading to NaNs
        pred_r0 = torch.clamp(pred_r0, -5.0, 5.0)

        if t_idx > 0:
            # Add noise for the next step, using alpha_t_prev and sigma_t_prev
            r_t = alpha_t_prev * pred_r0 + sigma_t_prev * torch.randn_like(r_t)
        else:
            r_t = pred_r0

    return r_t"""

with open(conditioned_infrastructure_path, 'w') as f:
    f.write(conditioned_file_content)

print(f"Successfully updated {conditioned_infrastructure_path}")

Successfully updated /content/butterflai/weeks/week_09/conditioned_infrastructure.py


In [15]:
# Keep - to set paths and load all modules needed

# Execute cell d9752246 to ensure paths are set and base modules are loaded
import os, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat

# Assuming repo_path is defined from earlier execution
# ── locate Week 08/09 artifacts (split across two folders) ─────────────────
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

# Correctly set search directories relative to repo_path
_search_dirs = [
    os.path.join(repo_path, "weeks", "week_09"),
    os.path.join(repo_path, "weeks", "week_08")
]

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path      = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)

_missing = [n for n in [
    "unconditioned_infrastructure.py",
    "conditioned_infrastructure.py",
    "diffusion_windows.parquet",
    "butterflAI_model.py",
    "official_model.npz",
] if _find(n, _search_dirs) is None]

if _missing:
    raise FileNotFoundError(
        f"Cannot locate {_missing} under weeks/week_08 or weeks/week_09. "
        f"Searched: {_search_dirs}"
    )

# Add the relevant directories to sys.path for imports
for _p in _search_dirs:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Week 08 reused machinery
from unconditioned_infrastructure import (
    make_cosine_schedule, ResidualDataset,
    DiffusionMLP, DiffusionLightning, sample,
)

# Import the conditioned_infrastructure module for later reloading
import conditioned_infrastructure

# ── data (same parquet as training) ────────────────────────────────────────
windows_df = pd.read_parquet(_parquet_path)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# Schedule arrays — used as placeholders at load time; the actual values
# come from the checkpoint's saved buffers.
T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Loaded {len(windows_df)} windows. Split sizes: "
      f"{windows_df['split'].value_counts().sort_index().to_dict()}")

Device: cpu
Loaded 373 windows. Split sizes: {'test': 86, 'train': 232, 'val': 55}


In [16]:
# Keep - to reload the conditioned_infrastructure module

# Execute cell 7d899036 to reload the conditioned_infrastructure module
import importlib

# Force reload the conditioned_infrastructure module
importlib.reload(conditioned_infrastructure)
print("Reloaded conditioned_infrastructure module.")

# Now, import the classes from the reloaded module
from conditioned_infrastructure import (
    ConditionalResidualDataset,
    ConditionalDiffusionMLP,
    ConditionalDiffusionLightning,
    sample_conditional,
)

# Diagnostic print to check ConditionalDiffusionMLP structure
print(f"ConditionalDiffusionMLP time_emb structure: {ConditionalDiffusionMLP(n_layers=1).time_emb}")

Reloaded conditioned_infrastructure module.
ConditionalDiffusionMLP time_emb structure: ModuleDict(
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
  )
)


---
## Load both checkpoints (Week 08 unconditional + Week 09 conditional)

The `load_from_checkpoint` pattern is the same as Week 08, with one
addition for the conditional module: `cond_means` and `cond_stds` are now
also among the buffers being restored, so the helper passes
placeholder zero/one tensors that the checkpoint overwrites.

After loading, both `lightning_uncond.bin_means` and
`lightning_cond.cond_means` should be **non-placeholder** values — that
is, *not* all zeros. The print statements at the end of the cell let you
verify this at a glance; if either looks all-zero or all-one, the
corresponding statistics were not saved with the checkpoint and every
sampler call downstream will silently de-normalize wrong.


In [17]:
# ── load both checkpoints ──────────────────────────────────────────────────
def _load_unconditional(ckpt_name):
    inner = DiffusionMLP(use_timestep_embedding=True)
    lm = DiffusionLightning.load_from_checkpoint(
        os.path.join(UNCONDITIONAL_CHECKPOINT_PATH, ckpt_name),
        model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

def _load_conditional(ckpt_name):
    # From the error messages: hidden_dim=256, t_embed_dim=128
    # The new error indicates t_hidden_dim should be 128 to match checkpoint
    inner = ConditionalDiffusionMLP(
        data_dim=15, cond_dim=4, # Updated cond_dim from 2 to 4
        hidden_dim=256, t_embed_dim=128, t_hidden_dim=128, # Corrected t_hidden_dim to 128
        n_layers=2 # Corrected from 3 to 2 to match checkpoint architecture
    )
    lm = ConditionalDiffusionLightning.load_from_checkpoint(
        os.path.join(CONDITIONAL_CHECKPOINT_PATH, ckpt_name),
        model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        cond_means=np.zeros(4, dtype=np.float32), # Updated size from 2 to 4
        cond_stds=np.ones(4, dtype=np.float32), # Updated size from 2 to 4
        map_location=device,
    )
    return lm.to(device).eval()

lightning_uncond = _load_unconditional("ckpt_unconditional_full.ckpt")
lightning_cond   = _load_conditional("ckpt_conditional.ckpt")

print("Loaded both modules.")
print(f"  uncond bin_means[:3]  = {lightning_uncond.bin_means[:3].cpu().numpy()}  "
      f" (should NOT be all zeros)")
print(f"  cond   bin_means[:3]  = {lightning_cond.bin_means[:3].cpu().numpy()}")
print(f"  cond   cond_means     = {lightning_cond.cond_means.cpu().numpy()}  "
      f" (should NOT be all zeros)")
print(f"  cond   cond_stds      = {lightning_cond.cond_stds.cpu().numpy()}  "
      f" (should NOT be all ones)")

RuntimeError: Error(s) in loading state_dict for ConditionalDiffusionLightning:
	size mismatch for cond_means: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([4]).
	size mismatch for cond_stds: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([4]).
	size mismatch for model.mlp.0.weight: copying a param with shape torch.Size([256, 145]) from checkpoint, the shape in current model is torch.Size([256, 147]).

---
## Task 49 (loaded) — re-verify t-sensitivity and cond-sensitivity on the loaded model

The training notebook ran both sanity checks on a fresh model. This is the
same pair of checks on the *loaded* conditional model. If something went
wrong during serialization or loading (the conditioning concatenation got
disconnected, the timestep embedding's state did not transfer correctly,
the cond statistics buffers are wrong), one of these will catch it.


In [8]:
import torch

# Task 49 (loaded) — sanity checks on the conditional model after load.

torch.manual_seed(1)
r_t_fixed = torch.randn(15, device=device)
t_values  = torch.tensor([0, T//4, T//2, 3*T//4, T-1], dtype=torch.long, device=device)
cond_zero = torch.zeros(2, device=device)

# t-sensitivity (hold r_t, cond fixed, vary t)
r_t_batch  = repeat(r_t_fixed, "d -> n d", n=5)
cond_batch = repeat(cond_zero, "d -> n d", n=5)
with torch.no_grad():
    out_t = lightning_cond.model(r_t_batch, t_values, cond_batch)
max_off_t = (out_t[None] - out_t[:, None]).norm(dim=-1).max().item()

# cond-sensitivity (hold r_t, t fixed, vary cond over ±2 in normalized space)
cond_values = torch.tensor(
    [[-2., -2.], [-1., 1.], [0., 0.], [1., -1.], [2., 2.]],
    device=device,
)
t_fixed = torch.full((5,), T // 2, dtype=torch.long, device=device)
r_t_batch = repeat(r_t_fixed, "d -> n d", n=5)
with torch.no_grad():
    out_c = lightning_cond.model(r_t_batch, t_fixed, cond_values)
max_off_c = (out_c[None] - out_c[:, None]).norm(dim=-1).max().item()

print(f"loaded model t-sensitivity   : max off-diagonal L2 = {max_off_t:.4f}")
print(f"loaded model cond-sensitivity: max off-diagonal L2 = {max_off_c:.4f}")
# Both should be visibly non-zero.


loaded model t-sensitivity   : max off-diagonal L2 = 2.3811
loaded model cond-sensitivity: max off-diagonal L2 = 2.0996


---
## Task 53 — Conditional sample visualization

Pick a small set of conditioning vectors spanning interesting points in
(area_smoothed, mu_universal) space, generate samples conditioned on each,
and plot the results side-by-side. This is the qualitative payoff of
training a conditional model: the generated residuals should look
visibly different across the cond values, in ways that make some kind
of physical sense.

The cleanest choice for "interesting points": three or four real
validation windows from `windows_df` covering distinct phases of
distinct cycles — say, an early-phase window from a tall cycle, a
mid-phase window from a tall cycle, an early-phase window from a short
cycle, etc. Reading the conditioning straight out of `windows_df` (then
normalizing through `lightning_cond.cond_means` / `cond_stds`) means you
are sampling at exactly the operating points the model will be asked to
handle in the NLL evaluation, which is more meaningful than synthetic
conditioning.

For each chosen validation window, generate `N_PER_COND = 50` samples
and plot their mean and ±1σ band as a function of bin latitude. Overlay
the *actual* held-out residual for that window in a contrasting colour.
The sampled distribution does not need to be perfectly centered on the
actual residual — the residual is one draw from a noisy process — but
the actual residual should generally lie inside or near the sampled band.


In [13]:
N_PER_COND = 50

# Compute physical residual for each chosen validation window.
HIST_COLS = [f"hist_emp_{j:02d}" for j in range(15)]
PAR_COLS  = [f"hist_par_{j:02d}" for j in range(15)]

val_df = windows_df[windows_df["split"] == "val"].reset_index(drop=True)
chosen_idx = [0, len(val_df)//4, len(val_df)//2, 3*len(val_df)//4]
chosen = val_df.iloc[chosen_idx].reset_index(drop=True)

# Stack cond and normalize using the loaded module's cond_means/cond_stds.
# Updated to include 'amplitude' and 'model_sigma'
cond_raw  = torch.tensor(chosen[["area_smoothed", "mu_universal", "amplitude", "model_sigma"]].to_numpy(np.float32),
                           device=device)
cond_norm = (cond_raw - lightning_cond.cond_means) / lightning_cond.cond_stds

# Generate N_PER_COND samples per window — repeat each cond row.
cond_batched = repeat(cond_norm, "n d -> (n k) d", k=N_PER_COND)
torch.manual_seed(0)
samples = sample_conditional(lightning_cond, cond_batched, device=device).cpu().detach().numpy()
samples = samples.reshape(len(chosen), N_PER_COND, 15)

# Actual held-out residuals.
emp = chosen[HIST_COLS].to_numpy(np.float32)
par = chosen[PAR_COLS].to_numpy(np.float32)
true_residuals = emp - par

# Plot 4 panels, each with sample mean ± σ band and the true residual overlaid.
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax_idx, idx in enumerate(range(len(chosen))):
    mu_s, sd_s = samples[idx].mean(0), samples[idx].std(0)
    axes[ax_idx].fill_between(BIN_CENTERS, mu_s - sd_s, mu_s + sd_s, alpha=0.3, color="C2")
    axes[ax_idx].plot(BIN_CENTERS, mu_s,                color="C2", lw=2, label="sample mean")
    axes[ax_idx].plot(BIN_CENTERS, true_residuals[idx], color="C0", lw=2, label="true residual")
    axes[ax_idx].axhline(0, color="k", linewidth=0.4)
    row = chosen.iloc[idx]
    axes[ax_idx].set_title(f"cyc {int(row['cycle'])} {row['hemisphere']}  "
                         f"τ={row['tau_center']:.2f}  area={row['area_smoothed']:.1f}")
    axes[ax_idx].set_xlabel("|latitude| (°)"); axes[ax_idx].legend(fontsize=8)
axes[0].set_ylabel("residual")
fig.suptitle("Conditional samples (mean ± σ over 50 draws) vs. held-out residual")
plt.tight_layout(); plt.show()


RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1

---
## Task 54 — Conditional distributional verification

Headline of the evaluation notebook. For each validation window, sample
`N_PER_WIN` conditional residuals at that window's conditioning, then
aggregate over all validation windows to form a "conditional sample
distribution" comparable to the actual held-out validation residual
distribution. The unconditional Week 08 model is the reference baseline:
sample `N_TOTAL_UNCOND ≈ N_PER_WIN × N_VAL_WINDOWS` unconditional
residuals (with no targeting at all) and treat that as the "what if we
ignored conditioning" comparator.

The diagnostics from Week 08's Task 43 — bin-wise mean, bin-wise std,
bin-bin covariance heatmap — applied three ways: training residuals,
conditional samples (per-window targeting), unconditional samples
(random). The expected reading:

- *If conditioning is working*, the conditional samples' bin-wise std
  should be **smaller** than the unconditional samples' (because each
  window's conditional distribution is narrower than the marginal), and
  the conditional samples' aggregate covariance should match the
  validation residuals' aggregate covariance *better* than the
  unconditional samples' does. The improvement may be small at this
  dataset scale — the NLL notebook will quantify it.
- *If conditioning is not working*, the conditional and unconditional
  comparisons will look nearly identical. That's the same failure-mode
  signature the cond-sensitivity check at the top of this notebook
  is designed to catch *before* you spend time here.
- *If conditioning is working but the trained model is overconfident*,
  the conditional std will be visibly *narrower* than the validation
  residuals' actual per-window variability — the model produces
  residuals that look like noise-free predictions when in fact every
  real window has a genuine spread of residuals around the conditional
  mean. This is real and worth flagging if you see it.


In [14]:
N_PER_WIN = 20
HIST_COLS = [f"hist_emp_{j:02d}" for j in range(15)]
PAR_COLS  = [f"hist_par_{j:02d}" for j in range(15)]

val_df = windows_df[windows_df["split"] == "val"].reset_index(drop=True)
train_df = windows_df[windows_df["split"] == "train"].reset_index(drop=True)
N_TOTAL = len(val_df) * N_PER_WIN

# 1. Training residuals (physical units) for the reference.
emp = train_df[HIST_COLS].to_numpy(np.float32)
par = train_df[PAR_COLS].to_numpy(np.float32)
train_residuals_phys = emp - par
rng = np.random.default_rng(123)
train_arr = train_residuals_phys[rng.integers(0, len(train_residuals_phys), size=N_TOTAL)]

# 2. Held-out validation residuals (physical units).
emp_v = val_df[HIST_COLS].to_numpy(np.float32)
par_v = val_df[PAR_COLS].to_numpy(np.float32)
val_residuals_phys = emp_v - par_v   # shape (N_val, 15)

# 3. Conditional samples: per-window targeting.
# Updated to include 'amplitude' and 'model_sigma'
cond_raw  = torch.tensor(val_df[["area_smoothed", "mu_universal", "amplitude", "model_sigma"]].to_numpy(np.float32),
                           device=device)
cond_norm = (cond_raw - lightning_cond.cond_means) / lightning_cond.cond_stds
cond_batched = repeat(cond_norm, "n d -> (n k) d", k=N_PER_WIN)
torch.manual_seed(0)
samples_cond = sample_conditional(lightning_cond, cond_batched, device=device).cpu().detach().numpy()

# 4. Unconditional samples: N_TOTAL random draws.
torch.manual_seed(0)
samples_uncond = sample(lightning_uncond, batch_size=N_TOTAL, data_dim=15,
                          device=device).cpu().numpy()

# 5. Bin-wise statistics
def _binwise(arr):
    return arr.mean(0), arr.std(0), np.cov(arr, rowvar=False)

m_train, s_train, c_train = _binwise(train_arr)
m_val,   s_val,   c_val   = _binwise(val_residuals_phys)
m_cond,  s_cond,  c_cond  = _binwise(samples_cond)
m_unc,   s_unc,   c_unc   = _binwise(samples_uncond)

# Mean / std bar panels
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
w = BIN_WIDTH * 0.22
for ax, mt, mh, mc, mu, title in [
    (axes[0], m_train, m_val, m_cond, m_unc, "Bin-wise mean"),
    (axes[1], s_train, s_val, s_cond, s_unc, "Bin-wise std"),
]:
    ax.bar(BIN_CENTERS - 1.5 * w, mt, width=w, label="training",     color="C0", edgecolor="black", linewidth=0.3)
    ax.bar(BIN_CENTERS - 0.5 * w, mh, width=w, label="val (held-out)", color="C4", edgecolor="black", linewidth=0.3)
    ax.bar(BIN_CENTERS + 0.5 * w, mc, width=w, label="conditional",  color="C2", edgecolor="black", linewidth=0.3)
    ax.bar(BIN_CENTERS + 1.5 * w, mu, width=w, label="unconditional", color="C3", edgecolor="black", linewidth=0.3)
    ax.set_title(title); ax.set_xlabel("|latitude| (°)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Covariance heatmaps
vmax = max(np.abs(c_train).max(), np.abs(c_val).max(),
           np.abs(c_cond).max(),  np.abs(c_unc).max())
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, mat, title in [
    (axes[0], c_train, "Training"),
    (axes[1], c_val,   "Val (held-out)"),
    (axes[2], c_cond,  "Conditional samples"),
    (axes[3], c_unc,   "Unconditional samples"),
]:
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title); ax.set_xlabel("bin"); ax.set_ylabel("bin")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Bin-bin covariance: training vs. val vs. conditional vs. unconditional")
plt.tight_layout(); plt.show()

# Summary table
comparison = pd.DataFrame([
    {"row": "val (held-out)",         "binwise_mean_MSE_vs_val": 0.0,
     "binwise_std_MSE_vs_val": 0.0,
     "cov_frob_vs_val": 0.0},
    {"row": "conditional samples",    "binwise_mean_MSE_vs_val": float(np.mean((m_cond - m_val)**2)),
     "binwise_std_MSE_vs_val": float(np.mean((s_cond - s_val)**2)),
     "cov_frob_vs_val": float(np.linalg.norm(c_cond - c_val))},
    {"row": "unconditional samples",  "binwise_mean_MSE_vs_val": float(np.mean((m_unc  - m_val)**2)),
     "binwise_std_MSE_vs_val": float(np.mean((s_unc  - s_val)**2)),
     "cov_frob_vs_val": float(np.linalg.norm(c_unc  - c_val))},
])
print(comparison.to_string(index=False))

RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1

---
## Where Week 09 leaves us, and what the NLL notebook will measure

By the end of this notebook you have:

- A trained **conditional** diffusion model whose samples are visibly
  sensitive to the conditioning (Task 53) and that aggregates, at the
  distributional level, to something closer to held-out validation
  residuals than the unconditional baseline does (Task 54). The
  improvement may be small at this scale; the qualitative direction is
  what matters before quantification.
- A loaded, sanity-checked conditional checkpoint that the NLL notebook
  will use to compute the headline value-add metric.

What you have **not** done is the headline measurement: does combining
the classical ButterflAI density with conditional diffusion residuals
actually outperform the classical density alone, on `compute_global_nll`
applied to the held-out validation split? That comparison is its own
clean notebook, picking up where this one ends.

**What the NLL notebook will do.** For each validation window: read its
`(area, mu)`, build the classical density on `BIN_CENTERS`, draw N
conditional residual samples at that conditioning, add the samples to
the classical density (with appropriate non-negativity / normalization
handling), compute the per-window NLL of the actual emp histogram under
that mixture. Aggregate via `compute_global_nll` across all validation
windows. Compare against (a) classical alone and (b) classical + Week 08
unconditional samples to isolate how much the conditioning is worth.

That's the end-of-program payoff measurement. Everything in Week 09 was
infrastructure pointed at making that measurement possible.
